# Just using ModernBERT (no CEFR vocab) for comparison

In [1]:
from transformers import ModernBertForSequenceClassification
from transformers import set_seed

set_seed(42)

baseline_model = ModernBertForSequenceClassification.from_pretrained(
    "answerdotai/ModernBERT-base",
    num_labels=4,
)

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# Load Data

In [2]:
from datasets import Dataset, load_dataset

ds = load_dataset("json", data_files=r"C:\Users\coope\Documents\Learner_Writing_Classification\cefr_lexical_sophistication\cefr-lexical-sophistication-main\wai_data\cefr_vocab_data\train_cefr_LazyA1.jsonl", split="train")
# Keep only the columns you need
ds = ds.select_columns(["text", "label"])

# convert CEFR levels to integers in a new "labels" column
label_map = {"A2": 0, "B1": 1, "B2": 2, "C1": 3}
ds = ds.map(lambda x: {"labels": label_map[x["label"]]})

# remove the old "label" column
ds = ds.remove_columns(["label"])

print(ds)
ds[0]

Dataset({
    features: ['text', 'labels'],
    num_rows: 4195
})


{'text': "New Year is always an invite to establish goals for the next 365 days. There are some areas there I'm pleased like improving my idioms knowledge, writing, speaking, presenting scientific articles. There are also areas that I should improve like statistics, IA, algorithms and how to present a work I developed. So I need to have some strategies for achieving goals. 1 - Improving a new language: I need to study, daily, at least little time like 15 minutes. And this study includes writing, reading a lot. 2 - Presenting scientific articles: It's important to reach innovative findings, read, study and try to disseminate these new ideas. When I consider to be a presenter I need to study a lot. 3 - Improve my statistics knowledge: a challenge, but not impossible. As to achieve previous goals, I'll need to study sistematically and never give up. 4 - Understanding about algorithms - it's a new knowledge and very difficult to study for whom is not a specialty on this area but I think th

# Tokenize data

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

tokenized_ds = ds.map(lambda x: tokenizer(x["text"]))

# Remove text column and look at data (label + input IDs (for tokens))

In [4]:
# remove "text" column, as text has been tokenized and it is no longer needed
tokenized_ds = tokenized_ds.remove_columns(["text"])

# here is what one example looks like
tokenized_ds[0]

{'labels': 1,
 'input_ids': [50281,
  4257,
  10519,
  310,
  1900,
  271,
  19864,
  281,
  5100,
  7342,
  323,
  253,
  1735,
  23412,
  1897,
  15,
  1707,
  403,
  690,
  3672,
  627,
  309,
  1353,
  13864,
  751,
  11138,
  619,
  22467,
  3056,
  3640,
  13,
  4028,
  13,
  8288,
  13,
  15250,
  8249,
  7774,
  15,
  1707,
  403,
  671,
  3672,
  326,
  309,
  943,
  3157,
  751,
  9990,
  13,
  32815,
  13,
  11333,
  285,
  849,
  281,
  1246,
  247,
  789,
  309,
  3715,
  15,
  1893,
  309,
  878,
  281,
  452,
  690,
  8130,
  323,
  17170,
  7342,
  15,
  337,
  428,
  3173,
  40037,
  247,
  747,
  3448,
  27,
  309,
  878,
  281,
  1263,
  13,
  5312,
  13,
  387,
  1878,
  1652,
  673,
  751,
  1458,
  2909,
  15,
  1244,
  436,
  1263,
  3797,
  4028,
  13,
  4361,
  247,
  2257,
  15,
  374,
  428,
  23952,
  272,
  8249,
  7774,
  27,
  733,
  434,
  1774,
  281,
  3986,
  16694,
  4342,
  13,
  1239,
  13,
  1263,
  285,
  1611,
  281,
  22016,
  366,
  841,
  747

In [5]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
import torch

set_seed(42)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

args = TrainingArguments(
    save_strategy="no",
    output_dir="./modernbert-only",
    seed=42,
    learning_rate=1e-05,
    num_train_epochs=3,
)

trainer = Trainer(
    model=baseline_model, 
    train_dataset=tokenized_ds, 
    data_collator=data_collator,
    args=args
)

In [6]:
trainer.train()

Step,Training Loss
500,0.839100
1000,0.635600
1500,0.506500


TrainOutput(global_step=1575, training_loss=0.650881616501581, metrics={'train_runtime': 857.3605, 'train_samples_per_second': 14.679, 'train_steps_per_second': 1.837, 'total_flos': 2609731479207984.0, 'train_loss': 0.650881616501581, 'epoch': 3.0})

In [7]:
from datasets import load_dataset
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
import plotly.graph_objects as go
import numpy as np
import torch

# Load and preprocess validation data
val_ds = load_dataset("json", data_files=r"C:\Users\coope\Documents\Learner_Writing_Classification\cefr_lexical_sophistication\cefr-lexical-sophistication-main\wai_data\cefr_vocab_data\val_cefr_LazyA1.jsonl", split="train")
val_ds = val_ds.select_columns(["text", "label"])  # removed affix_bow

label_map = {"A2": 0, "B1": 1, "B2": 2, "C1": 3}
id_to_label = {0: "A2", 1: "B1", 2: "B2", 3: "C1"}

val_ds = val_ds.map(lambda x: {"labels": label_map[x["label"]]})
val_ds = val_ds.remove_columns(["label"])

# Tokenize
tokenized_val_ds = val_ds.map(lambda x: tokenizer(x["text"]))
tokenized_val_ds = tokenized_val_ds.remove_columns(["text"])

# Get predictions
baseline_model.eval()  # changed from new_model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
baseline_model.to(device)  # changed from new_model

all_preds = []
all_labels = []

with torch.no_grad():
    for example in tokenized_val_ds:
        input_ids = torch.tensor([example["input_ids"]]).to(device)
        attention_mask = torch.tensor([example["attention_mask"]]).to(device)
        # removed affix_bow
        
        outputs = baseline_model(input_ids=input_ids, attention_mask=attention_mask)  # changed from new_model, removed affix_bow
        pred = torch.argmax(outputs.logits, dim=-1).item()
        
        all_preds.append(pred)
        all_labels.append(example["labels"])

# Calculate metrics
precision, recall, f1, support = precision_recall_fscore_support(all_labels, all_preds, average=None)
labels = ["A2", "B1", "B2", "C1"]

print("Per-class metrics:")
for i, label in enumerate(labels):
    print(f"{label}: Precision={precision[i]:.3f}, Recall={recall[i]:.3f}, F1={f1[i]:.3f}")

# Overall metrics
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro")
print(f"\nMacro average: Precision={precision_macro:.3f}, Recall={recall_macro:.3f}, F1={f1_macro:.3f}")

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
cm_normalized = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

annotations = []
for i in range(len(labels)):
    row = []
    for j in range(len(labels)):
        row.append(f"{cm_normalized[i][j]*100:.1f}%<br>({cm[i][j]})")
    annotations.append(row)

fig = go.Figure(data=go.Heatmap(
    z=cm_normalized,
    x=labels,
    y=labels,
    colorscale=[[0, "white"], [1, "black"]],
    text=annotations,
    texttemplate="%{text}",
    textfont={"size": 12},
    hovertemplate="True: %{y}<br>Predicted: %{x}<br>Proportion: %{z:.2f}<extra></extra>"
))

fig.update_layout(
    title="ModernBERT Only",  # updated title
    xaxis_title="Predicted",
    yaxis_title="True",
    xaxis={"side": "bottom"},
    yaxis={"autorange": "reversed"},
    width=500,
    height=500
)

fig.show()

Per-class metrics:
A2: Precision=0.803, Recall=0.663, F1=0.726
B1: Precision=0.658, Recall=0.624, F1=0.641
B2: Precision=0.523, Recall=0.712, F1=0.603
C1: Precision=0.600, Recall=0.383, F1=0.468

Macro average: Precision=0.646, Recall=0.595, F1=0.609


# Calculate QWK

In [8]:
# %%
from sklearn.metrics import cohen_kappa_score

qwk = cohen_kappa_score(all_labels, all_preds, weights="quadratic")
print(f"Quadratic Weighted Kappa: {qwk:.3f}")

Quadratic Weighted Kappa: 0.759
